# Amyloid Pipeline — Full Run

Runs the end-to-end pipeline (grid search → ensemble CV → benchmark) and
displays **AUROC, PR, confusion matrix** and other metrics.

Each execution gets a **`run_id` = `yymmdd_hhmmss`** and writes to two folders:

```
results/<run_id>/training/    # OOF residue metrics + per-residue/per-window scores + plots + ensemble
results/<run_id>/benchmark/   # sequence metrics + per-residue/per-window scores + plots
```

> **Prerequisite — build the embedding cache first, in a terminal:**
> ```
> python scripts/01_generate_embeddings.py
> ```
> Embedding generation needs `torch` + `transformers` and is kept OUT of this notebook on
> purpose: loading PyTorch (ProtT5/ESM2) inside this kernel — which also loads TensorFlow —
> can crash the kernel via a native runtime conflict. This notebook only *reads* the cache.
> Everything else needs `tensorflow`, `scikit-learn`, `pandas`, `matplotlib` (see `requirements.txt`).


## Setup & `run_id`

In [ ]:
import sys, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make `src` importable whether the notebook runs from repo root or notebooks/
_here = Path.cwd()
_root = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(_root))

from src.utils.config import load_config, resolve_path, PROJECT_ROOT
from src.utils.seed import set_global_seed
from src.utils.io import write_json

ROOT = PROJECT_ROOT
cfg = load_config()
set_global_seed(cfg.get("seed", 42))

RUN_ID = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
run_dir   = ROOT / cfg["paths"]["results_dir"] / RUN_ID
train_dir = run_dir / "training"
bench_dir = run_dir / "benchmark"
for d in (train_dir / "plots", bench_dir / "plots"):
    d.mkdir(parents=True, exist_ok=True)

print("RUN_ID    :", RUN_ID)
print("training  :", train_dir)
print("benchmark :", bench_dir)

## Run configuration

Tune these for a quick smoke run vs. a full run. Grid search is **off by default**
(uses each model's first grid combo); set `RUN_GRIDSEARCH = True` to tune.

In [ ]:
RUN_GRIDSEARCH = True                  # True -> tune hyperparameters (slow)
LIMIT          = None                   # e.g. 60 -> cap #training peptides for a smoke run
MODELS         = cfg["gridsearch"]["models"]   # e.g. ["fnn", "cnn"] to go faster
EPOCHS         = cfg["train"]["epochs"]        # e.g. 8 for a quick run

cfg["gridsearch"]["models"] = MODELS
cfg["train"]["epochs"]      = EPOCHS
# Route grid-search / best-config artifacts into this run's training folder.
cfg["paths"]["results_dir"] = str(train_dir)

print("models   :", MODELS)
print("epochs   :", EPOCHS, "| gridsearch:", RUN_GRIDSEARCH, "| limit:", LIMIT)

## Stage 1 — Load data & verify embedding cache

> Prerequisite: run `python scripts/01_generate_embeddings.py` in a terminal **once** to
> build the cache. This cell only *reads* the cache and will stop if anything is missing —
> it never loads PyTorch in this kernel (that can crash the TensorFlow kernel).


In [ ]:
from src.data.dataset import load_training_peptides, load_benchmark, assert_no_leakage
from src.data.embeddings import EmbeddingCache  # generation is done by scripts/01 (separate process)
from src.utils.windows import required_inference_keys

peptides  = load_training_peptides(resolve_path(cfg, "train_pos"), resolve_path(cfg, "train_neg"))
benchmark = load_benchmark(resolve_path(cfg, "classification_benchmark"))
peptides  = assert_no_leakage(peptides, benchmark, drop=True)
if LIMIT:
    peptides = peptides[:LIMIT]
print(f"{len(peptides)} training peptides | {len(benchmark)} benchmark sequences")

cache    = EmbeddingCache(cfg["embeddings"]["cache"])
# In full mode this is the benchmark sequences themselves; in standalone mode it expands
# to every sliding-window subsequence (embeddings.embed_windows_standalone).
all_seqs = [p.sequence for p in peptides] + required_inference_keys([b.sequence for b in benchmark], cfg)
missing  = cache.missing(all_seqs)

# Embeddings are generated OUT OF PROCESS by scripts/01. We deliberately do NOT generate
# them here: loading PyTorch (ProtT5/ESM2) inside this TensorFlow kernel can crash it
# (native runtime conflict). If anything is missing, stop with clear instructions.
if missing:
    raise RuntimeError(
        f"{len(missing)} sequence(s) missing embeddings. Generate them FIRST in a separate "
        f"terminal:\n    python scripts/01_generate_embeddings.py\n"
        f"(cache: {cfg['embeddings']['cache']}), then re-run this notebook from the top."
    )
print(f"all {len(all_seqs)} sequences present in cache -> skipping generation")


## Stage 2 — Hyperparameter search (optional)

In [ ]:
from src.training.grid_search import run_gridsearch, default_best_per_model

if RUN_GRIDSEARCH:
    best_per_model = run_gridsearch(cfg, cache, peptides)     # writes gridsearch.csv + best/
else:
    best_per_model = default_best_per_model(cfg)
    write_json(best_per_model, train_dir / "best_per_model.json")

pd.DataFrame([{"model": m, **v["best_hp"]} for m, v in best_per_model.items()])

## Stage 3 — Train ensemble (best arch × 5 folds) with out-of-fold predictions

In [ ]:
from src.training.ensemble import build_ensemble_cv

ensemble, oof, per_model_fold = build_ensemble_cv(cfg, cache, peptides, best_per_model)
ensemble.save(train_dir / "ensemble")
print(f"{len(ensemble.members)} ensemble members trained "
      f"({len(best_per_model)} architectures x {cfg['cv']['n_splits']} folds)")

## Training metrics — residue-level, out-of-fold (OOF)

In [ ]:
from src.evaluation.metrics import residue_metrics

thr = cfg["inference"]["residue_threshold"]

# Per-architecture CV means
cv_df = pd.DataFrame(per_model_fold)
cv_summary = cv_df.groupby("model")[["auroc", "auprc", "mcc", "f1", "precision", "recall"]].mean().round(3)

# Ensemble OOF residue metrics
ens_metrics = residue_metrics(oof["labels"], oof["scores"], oof["mask"], threshold=thr)
print("Ensemble OOF residue metrics:")
for k, v in ens_metrics.items():
    print(f"  {k:10s}: {v}")

# --- save metrics + per-residue/per-window scores (each training peptide == one window)
write_json({"ensemble_oof": ens_metrics,
            "per_model_fold": per_model_fold,
            "per_model_mean": cv_summary.reset_index().to_dict("records")},
           train_dir / "metrics.json")
cv_df.to_csv(train_dir / "per_model_fold_metrics.csv", index=False)
np.savez_compressed(train_dir / "oof_residue_scores.npz",
                    ids=np.array(oof["ids"]), scores=oof["scores"],
                    labels=oof["labels"], mask=oof["mask"])
cv_summary

### Plots — ROC, PR, confusion matrix (training OOF, residue level)

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

m = oof["mask"].astype(bool)
y_true  = oof["labels"][m].ravel().astype(int)
y_score = oof["scores"][m].ravel()
y_pred  = (y_score > thr).astype(int)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_true, y_score, ax=ax[0]);            ax[0].set_title("ROC (residue OOF)")
PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=ax[1]);     ax[1].set_title("PR (residue OOF)")
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax[2], colorbar=False,
                                        display_labels=["non-core", "core"]); ax[2].set_title("Confusion (residue OOF)")
plt.tight_layout()
fig.savefig(train_dir / "plots" / "training_metrics.png", dpi=120)
plt.show()

## Stage 4 — Benchmark evaluation (sliding window + >0.5 / run>10 rule)

In [ ]:
from src.evaluation.sliding import score_sequence_detailed
from src.evaluation.classify import classify_profile, sequence_score

min_run = cfg["inference"]["min_consecutive"]
profiles, windows_out = {}, {}
y_true_seq, y_pred_seq, y_score_seq = [], [], []

for b in benchmark:
    profile, windows = score_sequence_detailed(b.sequence, ensemble, cache, cfg)
    label, run, pos_mask = classify_profile(profile, thr, min_run)
    sscore = sequence_score(profile, min_run)   # continuous score consistent with the rule
    y_true_seq.append(b.label); y_pred_seq.append(label); y_score_seq.append(sscore)
    profiles[b.id]    = {"sequence": b.sequence, "label_true": b.label, "label_pred": label,
                         "longest_run": int(run), "sequence_score": round(float(sscore), 4),
                         "profile": [round(float(x), 4) for x in profile]}
    windows_out[b.id] = windows

print(f"scored {len(benchmark)} benchmark sequences")

In [ ]:
from src.evaluation.metrics import sequence_metrics
from sklearn.metrics import roc_auc_score, average_precision_score

seq_m = sequence_metrics(y_true_seq, y_pred_seq)
seq_m["auroc"] = float(roc_auc_score(y_true_seq, y_score_seq)) if len(set(y_true_seq)) > 1 else float("nan")
seq_m["auprc"] = float(average_precision_score(y_true_seq, y_score_seq))
print("Benchmark sequence-level metrics:")
for k, v in seq_m.items():
    print(f"  {k:10s}: {v}")

write_json({"sequence_level": seq_m,
            "inference": {"residue_threshold": thr, "min_consecutive": min_run,
                          "window_len": cfg["window_len"], "stride": cfg["inference"]["window_stride"]},
            "ensemble_members": len(ensemble.members)},
           bench_dir / "metrics.json")
write_json(profiles, bench_dir / "profiles.json")     # per-residue scores
write_json(windows_out, bench_dir / "windows.json")   # per-window scores
pd.DataFrame({"id": [b.id for b in benchmark], "true": y_true_seq,
              "pred": y_pred_seq, "score": y_score_seq}).to_csv(
    bench_dir / "sequence_predictions.csv", index=False)

### Plots — ROC, PR, confusion matrix (benchmark, sequence level)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[0]);        ax[0].set_title("ROC (sequence)")
PrecisionRecallDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[1]); ax[1].set_title("PR (sequence)")
ConfusionMatrixDisplay.from_predictions(y_true_seq, y_pred_seq, ax=ax[2], colorbar=False,
                                        display_labels=["NONAMYLOID", "AMYLOID"]); ax[2].set_title("Confusion (sequence)")
plt.tight_layout()
fig.savefig(bench_dir / "plots" / "benchmark_metrics.png", dpi=120)
plt.show()

### Per-residue score profile (example positive sequence, core regions shaded)

In [ ]:
pos = next((b for b in benchmark if b.label == 1 and b.core_regions), None)
if pos:
    prof = np.array(profiles[pos.id]["profile"])
    plt.figure(figsize=(12, 3))
    plt.plot(prof, label="residue score")
    plt.axhline(thr, color="r", ls="--", label=f"threshold {thr}")
    for r in pos.core_regions:
        plt.axvspan(int(r.get("start", 1)) - 1, int(r.get("end", 0)), color="orange", alpha=0.3)
    info = profiles[pos.id]
    plt.title(f"{pos.id}  pred={'AMYLOID' if info['label_pred'] else 'NONAMYLOID'}  longest_run={info['longest_run']}")
    plt.xlabel("residue"); plt.ylabel("score"); plt.legend(); plt.tight_layout()
    plt.savefig(bench_dir / "plots" / f"profile_{pos.id}.png", dpi=120)
    plt.show()
else:
    print("no annotated positive sequence in the benchmark")

## Optional — position benchmark (positives only): recall + region IoU

In [ ]:
from src.evaluation.metrics import region_iou

pos_bench = load_benchmark(resolve_path(cfg, "position_benchmark"))
need = cache.missing(required_inference_keys([b.sequence for b in pos_bench], cfg))
if need:
    raise RuntimeError(
        f"{len(need)} position-benchmark embedding(s) missing. Run "
        f"'python scripts/01_generate_embeddings.py' first, then re-run this notebook."
    )

hits, ious, pos_profiles = 0, [], {}
for b in pos_bench:
    profile, _ = score_sequence_detailed(b.sequence, ensemble, cache, cfg)
    label, run, pos_mask = classify_profile(profile, thr, min_run)
    hits += int(label == 1)
    if b.core_regions:
        ious.append(region_iou(pos_mask, b.core_regions))
    pos_profiles[b.id] = {"label_pred": label, "longest_run": int(run),
                          "profile": [round(float(x), 4) for x in profile]}

recall   = hits / len(pos_bench)
mean_iou = float(np.nanmean(ious)) if ious else None
print(f"position_benchmark: recall={recall:.3f} ({hits}/{len(pos_bench)}), mean region IoU={mean_iou}")
write_json({"recall": recall, "n": len(pos_bench), "mean_region_iou": mean_iou},
           bench_dir / "position_benchmark_metrics.json")
write_json(pos_profiles, bench_dir / "position_benchmark_profiles.json")


## Summary — saved artifacts

In [ ]:
print("RUN_ID:", RUN_ID, "\n")
for tag, d in (("TRAINING", train_dir), ("BENCHMARK", bench_dir)):
    print(f"{tag}: {d}")
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print("   ", p.relative_to(d))
    print()